# Soiling Detection para o ESP32 · treino binário (limpar × não limpar) com o dataset tratado 224

O modelo responde **"precisa limpar?"** e é exportado como **TFLite int8** para o ESP32 (esp32dev, sem PSRAM).

**Decisão no dispositivo, com faixa para o humano:**
`P(limpar) ≥ LIMIAR_LIMPAR` → aciona a limpeza · `P(limpar) ≤ LIMIAR_NAO_LIMPAR` → não faz nada · entre os dois → revisão humana.

**O que mudou nesta versão**

| | Versão anterior (v5) | Esta versão |
|---|---|---|
| Dados | dataset bruto, divisão train/val/test sorteada | **dataset tratado 224×224**; o teste é **separado no notebook** por grupo de foto (ou vem de um zip externo) |
| Pré-processamento | resize do original → JPEG → RGB565 | 224 → `IMG_SIZE` (112) → JPEG 85 → RGB565 (igual ao dispositivo) |
| Grupos | fotos quase idênticas (dHash) | dHash **+ foto original** (cópias do Roboflow e `synth__` juntas) |
| Vazamento | — | **confere se alguma foto do teste está no treino** |
| Validação | validação com early stopping por AUC | **sem validação durante o treino**: acurácia e loss do treino por época |
| Validação cruzada | — | **folds ajustáveis** (`N_FOLDS_TO_RUN`): acurácia e loss de treino e teste de cada fold; escolhem os limiares |
| Modelo final | melhor época da validação | **treino do zero com todo o dataset de treino** |
| Métricas | teste opcional | **célula de métricas do modelo final** no teste, float **e int8** (o que roda no ESP32) |
| Aumento de dados | rotações de 90° + ângulo aleatório | igual + espelhamento opcional (cores nunca alteradas) |
| Arquiteturas | `soilnet` 1x/2x e `mbv1_025` | iguais, com a checagem "Cabe no ESP32?" antes de treinar |

**Como usar:** ajuste a célula **1** e rode em ordem. A célula 14 (ONNX) é opcional e fica por último.

Dependências: `pip install tensorflow numpy pillow matplotlib scikit-learn pandas` (o Colab já tem).

## 1. Configuração (o único lugar que você precisa mexer)

In [ ]:
# ---------------- dados (já tratados pelo notebook de tratamento: 224x224, enquadrados) ----------------
TRAIN_PATH = r"/content/Datasetv6.zip"          # zip OU pasta, com as subpastas das duas classes
TEST_PATH  = None             # None = separa o teste dentro do notebook | ou zip/pasta de um teste já separado
TEST_FRAC  = 0.15             # (com TEST_PATH=None) fração das fotos REAIS que vira teste
WORK_DIR   = None                                   # None = /content/treino_esp32 (Colab) ou ao lado do zip
CLASS_FOLDERS = None          # None = descobre pelas pastas | ou ("NaoLimpar", "Limpar") = (não aciona, aciona)
SYNTH_PREFIXES = ("synth__", "civitai__")           # imagens geradas: nunca entram no teste dos folds

# ---------------- modelo ----------------
ARCH     = "soilnet"          # "soilnet" = CNN própria do zero | "mbv1_025" = MobileNetV1 0,25 pré-treinada
IMG_SIZE = 112                # 96 | 112 | 128 (confira a célula 6)
DROPOUT  = 0.2
SOIL_PRESET  = "2x"           # "1x" (~67 mil parâmetros) | "2x" (~136 mil) | "custom" (usa SOIL_BLOCKS)
SOIL_PRESETS = {
    "1x": [(16, 2), (32, 2), (32, 1), (64, 2), (64, 1), (64, 1), (128, 2), (128, 1), (160, 1)],
    "2x": [(24, 2), (48, 2), (48, 1), (96, 2), (96, 1), (96, 1), (184, 2), (184, 1), (232, 1)],
}
SOIL_STEM   = 8               # manter 8: o stem define o pico de RAM
SOIL_BLOCKS = SOIL_PRESETS["1x"]                     # só com SOIL_PRESET = "custom"
SOIL_WIDTH  = 1.0
if SOIL_PRESET != "custom":
    SOIL_BLOCKS = SOIL_PRESETS[SOIL_PRESET]

# ---------------- limites do ESP32 ----------------
ARENA_BUDGET_KB = 96          # TENSOR_ARENA_BYTES do firmware
FLASH_BUDGET_KB = 300

# ---------------- pré-processamento (igual ao dispositivo) ----------------
JPEG_QUALITY    = 85          # 1..95, igual ao backend
SIMULATE_RGB565 = True

# ---------------- treino (sem validação: épocas fixas) ----------------
BATCH            = 16
USE_CLASS_WEIGHT = True
EPOCHS_SOILNET   = 450        # soilnet: épocas do zero
EPOCHS_HEAD      = 15         # mbv1_025: fase 1 (só a cabeça)
EPOCHS_FINETUNE  = 25         # mbv1_025: fase 2 (rede inteira)
LR_SOILNET       = 3e-3
LR_HEAD          = 1e-3
LR_FINETUNE      = 1e-4
WEIGHT_DECAY     = 1e-4
PRINT_EVERY      = 1

# ---------------- aumento de dados (só no treino; cores nunca alteradas) ----------------
AUGMENT          = True
AUG_ROT90_COPIES = True       # cada foto de treino entra 4x: 0°, 90°, 180°, 270°
AUG_RANDOM_DEG   = 10         # + ângulo aleatório de -N a +N graus a cada época (0 = desliga)
AUG_FLIP         = True       # espelhamento horizontal aleatório

# ---------------- validação cruzada e limiares ----------------
N_SPLITS       = 5
N_FOLDS_TO_RUN = 0           # 0 = pula (limiares 0,5) | 1..N_SPLITS
DHASH_MAX_DIST = 5            # fotos com dHash a essa distância ou menos ficam no mesmo grupo
PRECISAO_LIMPAR     = 0.75    # das vezes que aciona, quantas devem ser placa que precisava mesmo
PRECISAO_NAO_LIMPAR = 0.85    # das vezes que diz "não limpar", quantas devem estar certas

# ---------------- avaliação / exportação ----------------
N_BOOTSTRAP = 1000
EXPORT_ONNX = False           # célula 14 (opcional): instala tf2onnx e exporta o modelo float em ONNX
SEED = 42
QUICK = False                 # True = teste rápido (poucas épocas, 1 fold)
if QUICK:
    EPOCHS_SOILNET, EPOCHS_HEAD, EPOCHS_FINETUNE, N_FOLDS_TO_RUN, N_BOOTSTRAP = 2, 1, 1, min(N_FOLDS_TO_RUN, 1), 100

## 2. Imports e funções auxiliares

In [ ]:
import csv, datetime as dt, hashlib, json, math, random, re, shutil, sys, time, unicodedata, zipfile
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold

import tensorflow as tf
import keras
from keras import layers

IN_COLAB = "google.colab" in sys.modules
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| Colab" if IN_COLAB else "| local")
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)

erros_cfg = []
if not (1 <= JPEG_QUALITY <= 95): erros_cfg.append("JPEG_QUALITY deve estar entre 1 e 95 (o backend usa 85)")
if IMG_SIZE not in (96, 112, 128, 160): erros_cfg.append("IMG_SIZE: use 96, 112 ou 128")
if ARCH not in ("soilnet", "mbv1_025"): erros_cfg.append("ARCH: use 'soilnet' ou 'mbv1_025'")
if ARCH == "soilnet" and SOIL_PRESET not in (*SOIL_PRESETS, "custom"): erros_cfg.append("SOIL_PRESET inválido")
if not (0 <= N_FOLDS_TO_RUN <= N_SPLITS): erros_cfg.append("N_FOLDS_TO_RUN deve estar entre 0 e N_SPLITS")
if not (0.5 <= PRECISAO_LIMPAR < 1 and 0.5 <= PRECISAO_NAO_LIMPAR < 1): erros_cfg.append("PRECISAO_* entre 0.5 e 0.99")
if erros_cfg:
    raise ValueError("Corrija a célula 1:\n  - " + "\n  - ".join(erros_cfg))

WORK = Path(WORK_DIR) if WORK_DIR else (Path("/content/treino_esp32") if IN_COLAB else Path(TRAIN_PATH).parent / "treino_esp32")
WORK.mkdir(parents=True, exist_ok=True)
RUN_NAME = f"{ARCH}{'_' + SOIL_PRESET if ARCH == 'soilnet' else ''}_{IMG_SIZE}_{dt.datetime.now():%Y%m%d_%H%M}"
OUT = WORK / "modelos" / RUN_NAME; OUT.mkdir(parents=True, exist_ok=True)
print("resultados em:", OUT)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def norm(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().strip()

def device_like(path):
    # mesmo caminho que a foto faz até o ESP32: (224 tratada) -> IMG_SIZE -> JPEG -> RGB565 -> RGB888
    with Image.open(path) as im:
        img = ImageOps.exif_transpose(im).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    buf = BytesIO(); img.save(buf, "JPEG", quality=JPEG_QUALITY)
    x = np.asarray(Image.open(BytesIO(buf.getvalue())).convert("RGB"), dtype=np.uint8)
    if SIMULATE_RGB565:
        r, g, b = x[..., 0] >> 3, x[..., 1] >> 2, x[..., 2] >> 3
        x = np.stack([(r << 3) | (r >> 2), (g << 2) | (g >> 4), (b << 3) | (b >> 2)], -1).astype(np.uint8)
    return x

def dhash_bits(path):
    with Image.open(path) as im:
        g = np.asarray(ImageOps.exif_transpose(im).convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()

def source_id(path):
    s = Path(path).stem
    if s.lower().startswith("synth__"):
        s = re.sub(r"__s\d+$", "", s[len("synth__"):])
        s = s.rsplit("__", 1)[0] if "__" in s else s
    s = re.sub(r"^(train|valid|test)_+", "", s, flags=re.I)
    s = re.sub(r"\.rf\.[0-9a-f]+$", "", s, flags=re.I)
    s = re.sub(r"_(jpe?g|png|bmp|webp)(__\w+)?$", "", s, flags=re.I)
    return s.lower()

# ---------- métricas binárias (positivo = ACIONA a limpeza = índice 1) ----------
def confusion(y, p):
    cm = np.zeros((2, 2), dtype=int)
    for t, q in zip(y, p):
        cm[int(t), int(q)] += 1
    return cm

def f1_macro(y, p):
    cm = confusion(y, p); f1s = []
    for i in range(2):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        f1s.append(2 * tp / (2 * tp + fp + fn) if tp else 0.0)
    return float(np.mean(f1s))

def auc_binary(y, score):
    y = np.asarray(y, bool); score = np.asarray(score, float)
    npos, nneg = y.sum(), (~y).sum()
    if npos == 0 or nneg == 0:
        return float("nan")
    order = score.argsort(); ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    for s in np.unique(score):
        m = score == s; ranks[m] = ranks[m].mean()
    return float((ranks[y].sum() - npos * (npos + 1) / 2) / (npos * nneg))

def log_loss(y, p_pos):
    p = np.clip(np.asarray(p_pos, float), 1e-6, 1 - 1e-6)
    return float(-np.mean(np.where(np.asarray(y) == 1, np.log(p), np.log(1 - p))))

## 3. Carregar o dataset, separar treino e teste, descobrir qual pasta ACIONA a limpeza e conferir vazamento

- Duplicatas exatas saem.
- Com `TEST_PATH = None`, o teste sai do próprio dataset **antes de qualquer treino**: `TEST_FRAC` das fotos reais,
  estratificado por classe e **por grupo** (mesma foto original ou quase idêntica). Sintéticas nunca vão para o teste,
  e as geradas a partir de uma foto do teste saem do treino. A divisão fica em `particao.csv`.
- **Grupos**: mesma foto original (nome) **ou** fotos quase idênticas (dHash) ficam juntas na validação cruzada.
- **Vazamento**: o teste só mede o modelo de verdade se nenhuma foto dele (nem cópia ou versão sintética) estiver no treino.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/placas_dados"     # onde guardar os zips enviados por upload (Colab)

def _has_images(folder):
    return folder.exists() and any(p.suffix.lower() in IMG_EXT for p in folder.rglob("*") if p.is_file())

def safe_extract(zip_path, dest, name):
    # confere o zip, copia para o disco local se estiver no Drive e só marca como pronto se extrair tudo
    zip_path = Path(zip_path)
    if str(zip_path).startswith("/content/drive"):
        local = Path("/content") / f"_copia_{name}.zip"
        print(f"[{name}] copiando {zip_path.name} do Drive para o disco local...")
        shutil.copy(zip_path, local); zip_path = local
    try:
        with zipfile.ZipFile(zip_path) as z:
            ruim = z.testzip()
            if ruim:
                raise zipfile.BadZipFile(f"arquivo corrompido dentro do zip: {ruim}")
            tmp = dest.with_name(dest.name + "_tmp")
            shutil.rmtree(tmp, ignore_errors=True)
            z.extractall(tmp)
    except zipfile.BadZipFile as e:
        shutil.rmtree(dest.with_name(dest.name + "_tmp"), ignore_errors=True)
        raise RuntimeError(
            f"[{name}] o zip {zip_path.name} está corrompido ou incompleto ({e}).\n"
            "  - Abra o zip no seu computador para conferir se ele abre inteiro; se não, compacte a pasta de novo.\n"
            "  - Uploads grandes pelo navegador costumam ser cortados: prefira enviar o zip ao Google Drive\n"
            "    (pelo site do Drive) e apontar o caminho na célula 1, ex.: /content/drive/MyDrive/Datasetv6_224.zip\n"
            f"  - Se o arquivo veio de {DRIVE_DIR}, apague essa cópia para o notebook pedir de novo.") from None
    if not _has_images(tmp):
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError(f"[{name}] o zip {zip_path.name} não tem imagens.")
    shutil.rmtree(dest, ignore_errors=True)
    tmp.rename(dest)
    (dest / ".extraido").write_text(zip_path.name)
    print(f"[{name}] {zip_path.name} conferido e extraído em {dest}")
    return dest

def get_root(path, name):
    dest = WORK / f"dados_{name}"
    if (dest / ".extraido").exists() and _has_images(dest):     # só reaproveita extração completa
        print(f"[{name}] usando os dados já extraídos em {dest} (apague a pasta para trocar de dataset)")
        return dest
    shutil.rmtree(dest, ignore_errors=True)                      # descarta extração incompleta de antes
    path = Path(str(path))
    if not path.exists():
        cands = [Path("/content") / Path(str(path).replace("\\", "/")).name, Path(DRIVE_DIR) / path.name]
        path = next((c for c in cands if c.exists()), path)
    if not path.exists() and IN_COLAB:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            salvos = sorted(Path(DRIVE_DIR).glob(f"{name}__*.zip")) if Path(DRIVE_DIR).exists() else []
            if salvos:
                path = salvos[-1]; print(f"[{name}] usando o upload salvo no Drive: {path}")
        except Exception as e:
            print("não consegui montar o Drive:", e)
    if not path.exists():
        if not IN_COLAB:
            raise FileNotFoundError(f"[{name}] não encontrado: {path}. Ajuste a célula 1.")
        from google.colab import files
        print(f"[{name}] não encontrado. Selecione o ZIP de {name} (para arquivos grandes, prefira o Drive):")
        up = files.upload()
        if not up:
            raise RuntimeError(f"[{name}] nenhum arquivo enviado.")
        fname = next(iter(up)); path = Path("/content") / fname
        if not path.exists() or path.stat().st_size != len(up[fname]):
            path.write_bytes(up[fname])
        if path.suffix.lower() != ".zip":
            raise ValueError(f"[{name}] envie um .zip (recebi {fname}).")
        safe_extract(path, dest, name)                          # confere ANTES de guardar a cópia no Drive
        try:
            Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
            shutil.copy(path, Path(DRIVE_DIR) / f"{name}__{fname}")
            print(f"[{name}] cópia salva no Drive: {Path(DRIVE_DIR) / f'{name}__{fname}'}")
        except Exception as e:
            print(f"[{name}] não consegui salvar no Drive ({e}); o arquivo fica só nesta sessão.")
        return dest
    if path.is_dir():
        return path
    return safe_extract(path, dest, name)

def list_images(root):
    return [(p, p.parent.name) for p in sorted(root.rglob("*"))
            if p.is_file() and p.suffix.lower() in IMG_EXT and "__MACOSX" not in p.parts]

raw_tr = list_images(get_root(TRAIN_PATH, "treino"))
raw_te = list_images(get_root(TEST_PATH, "teste")) if TEST_PATH else []
found = sorted({c for _, c in raw_tr}, key=norm)
print("pastas no dataset:", {c: sum(1 for _, k in raw_tr if k == c) for c in found})
if raw_te:
    print("pastas no teste externo:", {c: sum(1 for _, k in raw_te if k == c) for c in sorted({c for _, c in raw_te})})

# ---------- qual pasta ACIONA a limpeza? ----------
NEGATION = re.compile(r"(^|[^a-z])(nao|no|not|sem|without)([^a-z]|$)|^(nao|no|not|sem)")
def aciona(nome):
    n = norm(nome)
    if "limpar" in n or any(w in n for w in ("sujo", "suja", "sujeira", "dirty", "soiled", "dusty")):
        base = True
    elif any(w in n for w in ("limpo", "limpa", "clean")):
        base = False
    else:
        return None
    return (not base) if NEGATION.search(n) else base

if CLASS_FOLDERS:
    neg, pos = CLASS_FOLDERS
else:
    assert len(found) == 2, f"esperava 2 pastas de classe, achei {found}. Informe CLASS_FOLDERS."
    flags = {c: aciona(c) for c in found}
    if sorted(flags.values(), key=str) != [False, True]:
        raise ValueError(f"não consegui decidir qual pasta aciona a limpeza: {flags}. Informe CLASS_FOLDERS.")
    neg = next(c for c, f in flags.items() if f is False); pos = next(c for c, f in flags.items() if f is True)
LABELS = (re.sub(r"\W+", "_", norm(neg)).upper(), re.sub(r"\W+", "_", norm(pos)).upper())
print("\n" + "=" * 70)
print(f"  NÃO aciona (índice 0): pasta '{neg}' -> {LABELS[0]}")
print(f"  ACIONA     (índice 1): pasta '{pos}' -> {LABELS[1]}")
print("  CONFIRA! Se estiver invertido, defina CLASS_FOLDERS na célula 1.")
print("=" * 70 + "\n")

def build_recs(raw, name):
    recs, seen, dup, bad = [], set(), 0, 0
    for p, c in raw:
        if c not in (neg, pos):
            continue
        try:
            with Image.open(p) as im:
                im.verify()
            h = hashlib.md5(p.read_bytes()).hexdigest()
        except Exception:
            bad += 1; continue
        if h in seen:
            dup += 1; continue
        seen.add(h)
        recs.append({"path": p, "label": 1 if c == pos else 0, "md5": h, "source": source_id(p), "dhash": dhash_bits(p),
                     "synth": p.name.lower().startswith(tuple(x.lower() for x in SYNTH_PREFIXES))})
    print(f"[{name}] {len(recs)} imagens | duplicatas: {dup} | ilegíveis: {bad}")
    return recs

R_tr = build_recs(raw_tr, "dataset")
R_te = build_recs(raw_te, "teste") if raw_te else []

# ---------- grupos no treino: mesma foto original OU dHash próximo (union-find) ----------
parent = list(range(len(R_tr)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
first = {}
for i, r in enumerate(R_tr):
    if r["source"] in first:
        union(i, first[r["source"]])
    else:
        first[r["source"]] = i
Hb = np.stack([r["dhash"] for r in R_tr]).astype(np.int32)
for s0 in range(0, len(R_tr), 1000):
    blk = Hb[s0:s0 + 1000]
    dist_m = blk @ (1 - Hb).T + (1 - blk) @ Hb.T
    for i, j in zip(*np.nonzero(dist_m <= DHASH_MAX_DIST)):
        if i + s0 < j:
            union(i + s0, j)
GROUPS = np.array([find(i) for i in range(len(R_tr))])
# ---------- split interno treino / teste (por grupo, só fotos reais no teste) ----------
if not TEST_PATH:
    real = np.array([i for i, r in enumerate(R_tr) if not r["synth"]])
    k_te = max(2, round(1 / TEST_FRAC))
    sg = StratifiedGroupKFold(n_splits=k_te, shuffle=True, random_state=SEED)
    _, te_pos = next(sg.split(np.zeros(len(real)), [R_tr[i]["label"] for i in real], GROUPS[real]))
    te_idx = set(real[te_pos].tolist()); g_te = set(GROUPS[list(te_idx)])
    tr_keep = [i for i in range(len(R_tr)) if i not in te_idx and GROUPS[i] not in g_te]
    descartadas = len(R_tr) - len(te_idx) - len(tr_keep)
    R_te = [R_tr[i] for i in sorted(te_idx)]
    R_tr, GROUPS = [R_tr[i] for i in tr_keep], GROUPS[tr_keep]
    Hb = Hb[tr_keep]
    print(f"split interno: treino {len(R_tr)} | teste {len(R_te)} ({len(R_te) / max(len(real), 1):.1%} das fotos reais)"
          + (f" | {descartadas} sintéticas descartadas (geradas de fotos do teste)" if descartadas else ""))
with open(OUT / "particao.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["arquivo", "rotulo", "particao", "grupo", "sintetica"])
    for r, g in zip(R_tr, GROUPS):
        w.writerow([r["path"].name, LABELS[r["label"]], "treino", int(g), r["synth"]])
    for r in R_te:
        w.writerow([r["path"].name, LABELS[r["label"]], "teste", "", r["synth"]])
print("divisão salva em", OUT / "particao.csv")

mistos = [g for g in set(GROUPS) if len({R_tr[i]["label"] for i in np.where(GROUPS == g)[0]}) > 1]
print(f"grupos no treino: {len(set(GROUPS))} (de {len(R_tr)} imagens) | grupos com as duas classes: {len(mistos)}"
      + (" (pares limpo/sujo gerados ou rótulos a revisar)" if mistos else ""))

# ---------- vazamento treino -> teste ----------
md5_tr, src_tr = {r["md5"] for r in R_tr}, {r["source"] for r in R_tr}
Ht = np.stack([r["dhash"] for r in R_te]).astype(np.int32)
d_te = (Ht @ (1 - Hb).T + (1 - Ht) @ Hb.T).min(1)
vaz = [dict(arquivo_teste=r["path"].name,
            motivo="idêntica" if r["md5"] in md5_tr else ("mesma foto original" if r["source"] in src_tr else "quase idêntica (dHash)"))
       for r, d in zip(R_te, d_te) if r["md5"] in md5_tr or r["source"] in src_tr or d <= DHASH_MAX_DIST]
print(f"\nfotos do teste que também aparecem no treino: {len(vaz)}")
if vaz:
    display(pd.DataFrame(vaz).head(15))
    pd.DataFrame(vaz).to_csv(OUT / "vazamento_teste_no_treino.csv", index=False)
    print("ATENÇÃO: as métricas do teste ficarão otimistas. Remova essas fotos do treino (lista em vazamento_teste_no_treino.csv).")
if any(r["synth"] for r in R_te):
    print("ATENÇÃO: há imagens sintéticas no teste. O teste deve ter só fotos reais.")

tab = pd.DataFrame({
    "treino (reais)": pd.Series([LABELS[r["label"]] for r in R_tr if not r["synth"]]).value_counts(),
    "treino (sintéticas)": pd.Series([LABELS[r["label"]] for r in R_tr if r["synth"]], dtype=object).value_counts(),
    "teste": pd.Series([LABELS[r["label"]] for r in R_te]).value_counts(),
}).reindex(list(LABELS)).fillna(0).astype(int)
display(tab)

## 4. Carregar as imagens como o ESP32 vai vê-las

In [ ]:
t0 = time.time()
X_all = np.stack([device_like(r["path"]) for r in R_tr]); Y_all = np.array([r["label"] for r in R_tr], dtype=np.int64)
X_te = np.stack([device_like(r["path"]) for r in R_te]); Y_te = np.array([r["label"] for r in R_te], dtype=np.int64)
SYN = np.array([r["synth"] for r in R_tr])
print(f"treino {X_all.shape} | teste {X_te.shape} | {time.time() - t0:.0f}s")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for row in range(2):
    idx = np.where(Y_all == row)[0][:8]
    for col in range(8):
        ax = axes[row, col]; ax.axis("off")
        if col < len(idx):
            ax.imshow(X_all[idx[col]])
        if col == 0:
            ax.set_title(f"{LABELS[row]} ({'ACIONA' if row == 1 else 'não aciona'})", loc="left", fontsize=10)
plt.suptitle(f"Como o ESP32 vê: {IMG_SIZE}x{IMG_SIZE}, JPEG {JPEG_QUALITY}" + (", RGB565" if SIMULATE_RGB565 else ""))
plt.tight_layout(); plt.show()

## 5. Modelo e aumento de dados (rotações em graus, cores preservadas)

- `soilnet`: CNN compacta própria (depthwise-separable + atalhos residuais), recebe pixels 0..255 direto.
- `mbv1_025`: MobileNetV1 0,25 pré-treinada no ImageNet (fase 1 só a cabeça, fase 2 a rede inteira).
- Aumento só no treino: cópias giradas de 90° (`AUG_ROT90_COPIES`), ângulo aleatório pequeno (`AUG_RANDOM_DEG`,
  com reflexo nas bordas) e espelhamento. Fica fora do modelo exportado: não pesa no ESP32.

In [ ]:
def soil_ch(c):
    return max(8, int(round(c * SOIL_WIDTH / 8)) * 8)

def build_model(blocks=None):
    blocks = SOIL_BLOCKS if blocks is None else blocks
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="input")                  # pixels RGB 0..255
    x = inp if ARCH == "soilnet" else layers.Rescaling(1 / 127.5, offset=-1.0)(inp)
    backbone = None
    if ARCH == "mbv1_025":
        backbone = keras.applications.MobileNet(input_shape=(IMG_SIZE, IMG_SIZE, 3), alpha=0.25,
                                                include_top=False, weights="imagenet")
        x = backbone(x, training=False)
    else:
        x = layers.Conv2D(soil_ch(SOIL_STEM), 3, strides=2, padding="same", use_bias=False, name="stem")(x)
        x = layers.BatchNormalization(name="stem_bn")(x); x = layers.ReLU(name="stem_relu")(x)
        for k, (c, s) in enumerate(blocks):
            c = soil_ch(c); shortcut = x
            x = layers.DepthwiseConv2D(3, strides=s, padding="same", use_bias=False, name=f"b{k}_dw")(x)
            x = layers.BatchNormalization(name=f"b{k}_dw_bn")(x); x = layers.ReLU(name=f"b{k}_dw_relu")(x)
            x = layers.Conv2D(c, 1, use_bias=False, name=f"b{k}_pw")(x)
            x = layers.BatchNormalization(name=f"b{k}_pw_bn")(x)
            if s == 1 and shortcut.shape[-1] == c:
                x = layers.Add(name=f"b{k}_add")([x, shortcut])
            x = layers.ReLU(name=f"b{k}_relu")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(2, activation="softmax", name="probs")(x)               # [P(não limpar), P(limpar)]
    return keras.Model(inp, out, name=ARCH), backbone

_m, _ = build_model()
print(f"{ARCH}{' ' + SOIL_PRESET if ARCH == 'soilnet' else ''} @ {IMG_SIZE}px: {_m.count_params():,} parâmetros")
del _m

rot_layer = (layers.RandomRotation(AUG_RANDOM_DEG / 360.0, fill_mode="reflect", interpolation="bilinear")
             if AUGMENT and AUG_RANDOM_DEG > 0 else None)

def augment_one(x, y, w):
    if AUG_FLIP:
        x = tf.image.random_flip_left_right(x)
    if rot_layer is not None:
        x = rot_layer(x[None], training=True)[0]
    return x, y, w

def make_train_ds(X, Y):
    counts = np.bincount(Y, minlength=2)
    cw = ({i: float(len(Y) / (2 * max(c, 1))) for i, c in enumerate(counts)} if USE_CLASS_WEIGHT else {0: 1.0, 1: 1.0})
    W = np.array([cw[int(c)] for c in Y], dtype="float32")
    Y1h = np.eye(2, dtype="float32")[Y]
    if AUGMENT and AUG_ROT90_COPIES:
        X = np.concatenate([np.rot90(X, k, axes=(1, 2)) for k in range(4)])
        Y1h, W = np.tile(Y1h, (4, 1)), np.tile(W, 4)
    ds = tf.data.Dataset.from_tensor_slices((X.astype("float32"), Y1h, W))
    ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    if AUGMENT:
        ds = ds.map(augment_one, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE), math.ceil(len(X) / BATCH), cw

if AUGMENT:
    fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
    for row in range(2):
        img = X_all[np.where(Y_all == row)[0][0]]
        views = [img] + [np.rot90(img, k, axes=(0, 1)) for k in (1, 2, 3)]
        views += [augment_one(tf.constant(img, tf.float32), 0, 0)[0].numpy().clip(0, 255).astype(np.uint8) for _ in range(2)]
        for col, (v, t) in enumerate(zip(views, ["original", "90°", "180°", "270°", "aleatório", "aleatório"])):
            axes[row, col].imshow(v); axes[row, col].axis("off"); axes[row, col].set_title(f"{LABELS[row]} · {t}" if col == 0 else t, fontsize=9)
    plt.suptitle("Aumento de dados: rotações e espelhamento (cores preservadas)"); plt.tight_layout(); plt.show()

## 6. Cabe no ESP32? (antes de treinar)

Converte a rede **sem treino** para int8: o tamanho e a memória não dependem dos pesos. Com a `soilnet`, compara
com a `1x`. Se aparecer **NÃO CABE**, use `SOIL_PRESET="1x"` ou reduza `IMG_SIZE` e rode de novo a partir da célula 4.

In [ ]:
def to_tflite_int8(m, x_rep, export_dir):
    rep = x_rep[np.random.default_rng(SEED).permutation(len(x_rep))[:300]].astype("float32")
    try:
        m.export(str(export_dir))
        conv = tf.lite.TFLiteConverter.from_saved_model(str(export_dir))
    except Exception as e:
        print("export SavedModel falhou, usando from_keras_model:", type(e).__name__)
        conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = lambda: ([r[None]] for r in rep)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    return conv.convert()

def tflite_info(blob):
    it = tf.lite.Interpreter(model_content=blob); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    try:
        ops = it._get_ops_details()
    except Exception:
        ops = []
    arena = None
    if ops:
        Td = {t["index"]: t for t in it.get_tensor_details()}
        act = {int(t) for d in ops for t in d["outputs"]} | {int(i["index"])}
        first_k, last_k = {}, {}
        for k, d in enumerate(ops):
            for t in list(d["inputs"]) + list(d["outputs"]):
                if int(t) in act:
                    first_k.setdefault(int(t), k); last_k[int(t)] = k
        size = {t: int(np.prod(Td[t]["shape"])) * np.dtype(Td[t]["dtype"]).itemsize for t in act if t in Td}
        arena = max(sum(s for t, s in size.items() if first_k.get(t, 1e9) <= k <= last_k.get(t, -1)) for k in range(len(ops)))
    q = lambda d: {"shape": [int(v) for v in d["shape"]], "dtype": "int8",
                   "scale": float(d["quantization"][0]), "zero_point": int(d["quantization"][1])}
    return {"input": q(i), "output": q(o), "ops": sorted({d["op_name"] for d in ops} - {"DELEGATE"}),
            "arena_estimada_bytes": int(arena) if arena else None}

def memory_report(info, blob_len):
    arena_kb = (info["arena_estimada_bytes"] or 0) / 1024 + 10
    ok_a, ok_f = arena_kb <= ARENA_BUDGET_KB, blob_len / 1024 <= FLASH_BUDGET_KB
    print(f"modelo (flash): {blob_len / 1024:.1f} KB [{'ok' if ok_f else 'NÃO CABE'}, limite {FLASH_BUDGET_KB} KB]")
    print(f"arena (RAM):    ~{arena_kb:.0f} KB [{'ok' if ok_a else 'NÃO CABE'}, limite {ARENA_BUDGET_KB} KB]")
    print(f"buffers do firmware: imagem RGB565 {IMG_SIZE * IMG_SIZE * 2 / 1024:.0f} KB + JPEG recebido 32 KB")
    print("ops:", ", ".join(info["ops"]))
    return ok_a and ok_f

_m, _ = build_model()
_blob = to_tflite_int8(_m, X_all, OUT / "_sm_sem_treino")
cabe = memory_report(tflite_info(_blob), len(_blob))
if ARCH == "soilnet" and SOIL_PRESET != "1x":
    _m1, _ = build_model(SOIL_PRESETS["1x"]); _b1 = to_tflite_int8(_m1, X_all, OUT / "_sm_1x")
    print(f"\ncomparação com a 1x: parâmetros {_m1.count_params():,} -> {_m.count_params():,} | "
          f"modelo {len(_b1) / 1024:.1f} KB -> {len(_blob) / 1024:.1f} KB")
    shutil.rmtree(OUT / "_sm_1x", ignore_errors=True)
shutil.rmtree(OUT / "_sm_sem_treino", ignore_errors=True)
print("\n=> cabe no ESP32, pode treinar" if cabe else "\n=> NÃO CABE: use SOIL_PRESET='1x' ou reduza IMG_SIZE")

## 7. Função de treino (sem validação: só acurácia e loss do treino por época)

Épocas fixas, sem early stopping e sem escolher época por validação. `soilnet`: uma fase do zero com aquecimento do
learning rate. `mbv1_025`: fase 1 (cabeça) e fase 2 (rede inteira).

In [ ]:
class PrintEpoch(keras.callbacks.Callback):
    def __init__(self, name, phase, total):
        super().__init__(); self.name, self.phase, self.total, self.rows, self.t0 = name, phase, total, [], time.time()
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.rows.append({"fase": self.phase, "loss_treino": logs.get("loss"), "acc_treino": logs.get("accuracy")})
        if epoch == 0 or (epoch + 1) % PRINT_EVERY == 0 or epoch + 1 == self.total:
            print(f"[{self.name} · {self.phase}] época {epoch + 1:3d}/{self.total} | loss treino {logs.get('loss'):.4f} | "
                  f"acc treino {logs.get('accuracy') * 100:6.2f}% | {time.time() - self.t0:.0f}s")
            self.t0 = time.time()

def train_model(X, Y, name, seed=SEED):
    keras.utils.set_random_seed(seed)
    model, backbone = build_model()
    ds, steps, cw = make_train_ds(X, Y)
    print(f"[{name}] {len(X)} fotos ({np.bincount(Y, minlength=2).tolist()}) -> {steps * BATCH} amostras/época | "
          f"pesos {({LABELS[k]: round(v, 2) for k, v in cw.items()})}")
    if backbone is None:
        phases = [("do zero", EPOCHS_SOILNET, LR_SOILNET, None)]
    else:
        phases = [("cabeça", EPOCHS_HEAD, LR_HEAD, False), ("fine-tune", EPOCHS_FINETUNE, LR_FINETUNE, True)]
    hist, t0 = [], time.time()
    for phase, epochs, lr, train_bb in phases:
        if epochs <= 0:
            continue
        if backbone is not None:
            backbone.trainable = train_bb
        total = epochs * steps
        if backbone is None:
            warm = min(3 * steps, total // 4)
            sched = keras.optimizers.schedules.CosineDecay(lr * 0.1, max(1, total - warm), alpha=0.02,
                                                           warmup_target=lr, warmup_steps=warm)
        else:
            sched = keras.optimizers.schedules.CosineDecay(lr, total, alpha=0.05)
        model.compile(optimizer=keras.optimizers.AdamW(sched, weight_decay=WEIGHT_DECAY),
                      loss=keras.losses.CategoricalCrossentropy(),
                      metrics=[keras.metrics.CategoricalAccuracy(name="accuracy")])
        cb = PrintEpoch(name, phase, epochs)
        model.fit(ds, epochs=epochs, callbacks=[cb], verbose=0)
        hist += cb.rows
    h = pd.DataFrame(hist); h.insert(0, "epoca", np.arange(1, len(h) + 1))
    print(f"[{name}] tempo: {(time.time() - t0) / 60:.1f} min")
    return model, h

def predict_float(model, X):
    return model.predict(X.astype("float32"), verbose=0, batch_size=128)[:, 1]

def plot_train(hists, titles, marks=None):
    fig, axes = plt.subplots(1, len(hists), figsize=(4.8 * len(hists), 3.4), squeeze=False)
    for k, (ax, h, t) in enumerate(zip(axes[0], hists, titles)):
        ax.plot(h.epoca, h.loss_treino, c="#c53030", label="loss treino"); ax.set_xlabel("época"); ax.set_title(t, fontsize=9)
        a2 = ax.twinx(); a2.plot(h.epoca, h.acc_treino * 100, c="#2b6cb0", label="acc treino"); a2.set_ylim(0, 100); a2.grid(alpha=0.3)
        if marks:
            lt, at = marks[k]
            ax.scatter([h.epoca.iloc[-1]], [lt], marker="s", s=60, c="#c53030", edgecolor="k", zorder=5, label="loss teste")
            a2.scatter([h.epoca.iloc[-1]], [at * 100], marker="s", s=60, c="#2b6cb0", edgecolor="k", zorder=5, label="acc teste")
        if (h.fase == "fine-tune").any():
            ax.axvline(h.epoca[h.fase == "fine-tune"].iloc[0] - 0.5, c="gray", ls=":")
        if k == 0:
            h1, l1 = ax.get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels()
            fig.legend(h1 + h2, l1 + l2, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.1))
    plt.tight_layout(); plt.show()

## 8. Validação cruzada (folds ajustáveis) e calibração dos limiares

Roda os primeiros `N_FOLDS_TO_RUN` folds (fotos reais, por grupo; sintéticas só no treino de cada fold).
Cada fold mostra **acurácia e loss de treino e de teste**. As previsões dos folds calibram os dois limiares
(acionar / não acionar) sem tocar no seu teste.

In [ ]:
def calibrate(p_limpar, y, prec_limpar, prec_nao_limpar, min_n=5):
    ts = np.round(np.arange(0.05, 0.96, 0.01), 2)
    t_high = next((float(t) for t in ts if (p_limpar >= t).sum() >= min_n and (y[p_limpar >= t] == 1).mean() >= prec_limpar), None)
    t_low = next((float(t) for t in ts[::-1] if (p_limpar <= t).sum() >= min_n and (y[p_limpar <= t] == 0).mean() >= prec_nao_limpar), None)
    if t_high is None:
        t_high = 0.95; print(f"AVISO: nenhum limiar atinge precisão {prec_limpar:.0%} para limpar; usando 0,95")
    if t_low is None:
        t_low = 0.05; print(f"AVISO: nenhum limiar atinge precisão {prec_nao_limpar:.0%} para 'não limpar'; usando 0,05")
    if t_low >= t_high:
        t_low = t_high = round((t_low + t_high) / 2, 2)
    return t_low, t_high

idx_real, idx_syn = np.where(~SYN)[0], np.where(SYN)[0]
FOLDS = []
if N_FOLDS_TO_RUN > 0:
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for tr, va in skf.split(np.zeros(len(idx_real)), Y_all[idx_real], GROUPS[idx_real]):
        te_i = idx_real[va]; g_te = set(GROUPS[te_i])
        tr_i = np.concatenate([idx_real[tr], [i for i in idx_syn if GROUPS[i] not in g_te]]).astype(int)
        FOLDS.append((tr_i, te_i))

oof = np.full(len(Y_all), np.nan); cv_rows, cv_hists = [], []
for k, (tr, te) in enumerate(FOLDS[:N_FOLDS_TO_RUN], 1):
    print(f"\n{'=' * 78}\nFOLD {k}/{N_SPLITS} | treino {len(tr)} | teste {len(te)}\n{'=' * 78}")
    model, hist = train_model(X_all[tr], Y_all[tr], f"fold {k}", seed=SEED + k)
    p = predict_float(model, X_all[te]); oof[te] = p
    row = dict(fold=k, acc_treino=hist.acc_treino.iloc[-1], loss_treino=hist.loss_treino.iloc[-1],
               acc_teste=float(((p >= 0.5) == Y_all[te]).mean()), loss_teste=log_loss(Y_all[te], p),
               f1_teste=f1_macro(Y_all[te], (p >= 0.5).astype(int)), auc_teste=auc_binary(Y_all[te] == 1, p))
    cv_rows.append(row); cv_hists.append(hist)
    print(f"RESULTADO fold {k}: treino acc {row['acc_treino'] * 100:.2f}% loss {row['loss_treino']:.4f} | "
          f"teste acc {row['acc_teste'] * 100:.2f}% loss {row['loss_teste']:.4f} | AUC {row['auc_teste']:.3f}")
    del model; keras.backend.clear_session()

if cv_rows:
    df_cv = pd.DataFrame(cv_rows)
    resumo = df_cv.drop(columns="fold").agg(["mean", "std"])
    display(pd.concat([df_cv.set_index("fold"), resumo]).round(4))
    plot_train(cv_hists, [f"fold {k}" for k in df_cv.fold], [(r["loss_teste"], r["acc_teste"]) for r in cv_rows])
    m = ~np.isnan(oof)
    LIMIAR_NAO_LIMPAR, LIMIAR_LIMPAR = calibrate(oof[m], Y_all[m], PRECISAO_LIMPAR, PRECISAO_NAO_LIMPAR)
    if (df_cv.acc_treino - df_cv.acc_teste).mean() > 0.10:
        print("Treino bem acima do teste: OVERFITTING. Reduza épocas, aumente DROPOUT, use SOIL_PRESET='1x' ou mbv1_025.")
    elif df_cv.acc_treino.mean() < 0.80:
        print("Treino abaixo de 80%: SUBAJUSTE. Aumente épocas, use SOIL_PRESET='2x' ou mbv1_025.")
else:
    df_cv = None
    LIMIAR_NAO_LIMPAR = LIMIAR_LIMPAR = 0.5
    print("validação cruzada pulada: limiares em 0,50")
print(f"\nlimiares: P(limpar) >= {LIMIAR_LIMPAR:.2f} limpar | <= {LIMIAR_NAO_LIMPAR:.2f} não limpar | entre os dois: revisão humana")

## 9. Treino final (do zero, com todo o dataset de treino)

Modelo **novo**, sem reaproveitar pesos dos folds, treinado com **todas** as imagens de treino (reais +
sintéticas). As imagens de teste continuam fora.

In [ ]:
keras.backend.clear_session()
final_model, hist_final = train_model(X_all, Y_all, "modelo final", seed=SEED)
plot_train([hist_final], ["modelo final (100% do treino)"])
final_model.save(OUT / "model.keras")
print("salvo:", OUT / "model.keras")

## 10. Exportação para o ESP32

Gera em `OUT`: `model.tflite` (int8), `model_data.h`, `labels.json`, `model_card.json` (atualizado com as métricas
na célula 11).

In [ ]:
def tflite_predict(blob, x):
    it = tf.lite.Interpreter(model_content=blob); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    (si, zi), (so, zo) = i["quantization"], o["quantization"]
    out = []
    for img in x:
        it.set_tensor(i["index"], np.clip(np.round(img.astype("float32") / si + zi), -128, 127).astype(np.int8)[None])
        it.invoke()
        out.append((it.get_tensor(o["index"])[0].astype("float32") - zo) * so)
    return np.asarray(out)[:, 1]

blob = to_tflite_int8(final_model, X_all, OUT / "saved_model")
(OUT / "model.tflite").write_bytes(blob)
info = tflite_info(blob)
hexs = ",\n  ".join(", ".join(f"0x{b:02x}" for b in blob[i:i + 16]) for i in range(0, len(blob), 16))
(OUT / "model_data.h").write_text(
    "// GERADO pelo notebook de treino do ESP32. Não edite à mão.\n#pragma once\n#include <cstddef>\n"
    f"alignas(16) const unsigned char g_model_data[] = {{\n  {hexs}\n}};\nconst size_t g_model_data_len = {len(blob)};\n",
    encoding="utf-8")
(OUT / "labels.json").write_text(json.dumps(list(LABELS)), encoding="utf-8")
memory_report(info, len(blob))

## 11. Métricas de avaliação do modelo final (teste)

Avaliado em **float** (Keras) e em **int8** (exatamente o `model.tflite` que vai para o ESP32):
- acurácia e **loss** de treino e de teste;
- F1-macro, AUC, recall/precisão de "limpar", especificidade;
- **decisão com os limiares** (aciona / não aciona / revisão humana) e quanto erraria;
- matriz de confusão, curva ROC, distribuição de P(limpar) e intervalos de confiança de 95%.

In [ ]:
p_te_float = predict_float(final_model, X_te)
p_te_int8 = tflite_predict(blob, X_te)
p_tr_float = predict_float(final_model, X_all)

def decide(p, t_low, t_high):
    return np.where(p >= t_high, "limpar", np.where(p <= t_low, "nao_limpar", "revisao"))

def metrics(name, y, p):
    pred = (p >= 0.5).astype(int); cm = confusion(y, pred); tn, fp, fn, tp = cm.ravel()
    d = decide(p, LIMIAR_NAO_LIMPAR, LIMIAR_LIMPAR)
    fire, rest, rev = d == "limpar", d == "nao_limpar", d == "revisao"
    m = {"acuracia": float((pred == y).mean()), "loss": log_loss(y, p), "f1_macro": f1_macro(y, pred),
         "auc": auc_binary(y == 1, p), "recall_limpar": tp / max(tp + fn, 1), "precisao_limpar": tp / max(tp + fp, 1),
         "especificidade_nao_limpar": tn / max(tn + fp, 1), "matriz_confusao": cm.tolist(),
         "decisao": {"limiar_nao_limpar": LIMIAR_NAO_LIMPAR, "limiar_limpar": LIMIAR_LIMPAR,
                     "aciona": int(fire.sum()), "aciona_correto_pct": float((y[fire] == 1).mean()) if fire.any() else None,
                     "nao_aciona": int(rest.sum()), "nao_aciona_correto_pct": float((y[rest] == 0).mean()) if rest.any() else None,
                     "revisao_humana": int(rev.sum()), "cobertura_automatica": float(1 - rev.mean()),
                     "limpa_que_acionaria_pct": float(fire[y == 0].mean()) if (y == 0).any() else None,
                     "suja_ignorada_pct": float(rest[y == 1].mean()) if (y == 1).any() else None}}
    rng = np.random.default_rng(SEED); a, u = [], []
    for _ in range(N_BOOTSTRAP):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2:
            a.append(((p[i] >= 0.5) == y[i]).mean()); u.append(auc_binary(y[i] == 1, p[i]))
    m["ic95"] = {"acuracia": [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))],
                 "auc": [float(np.percentile(u, 2.5)), float(np.percentile(u, 97.5))]}
    fmt = lambda v: "n/d" if v is None else f"{v:.1%}"
    dd = m["decisao"]
    print(f"\n== {name} ==")
    print(f"   acurácia {m['acuracia'] * 100:.2f}% (IC95% {m['ic95']['acuracia'][0] * 100:.1f}–{m['ic95']['acuracia'][1] * 100:.1f}) | "
          f"loss {m['loss']:.4f} | F1-macro {m['f1_macro']:.4f} | AUC {m['auc']:.3f} "
          f"(IC95% {m['ic95']['auc'][0]:.3f}–{m['ic95']['auc'][1]:.3f})")
    print(f"   corte 0,5: recall limpar {m['recall_limpar']:.3f} | precisão limpar {m['precisao_limpar']:.3f} | "
          f"'não limpar' reconhecidas {m['especificidade_nao_limpar']:.3f}")
    print(f"   com limiares: aciona {dd['aciona']} (acerto {fmt(dd['aciona_correto_pct'])}) | não aciona {dd['nao_aciona']} "
          f"(acerto {fmt(dd['nao_aciona_correto_pct'])}) | revisão {dd['revisao_humana']} -> {dd['cobertura_automatica']:.0%} decididas sozinho")
    print(f"   limpas que acionariam: {fmt(dd['limpa_que_acionaria_pct'])} | sujas ignoradas: {fmt(dd['suja_ignorada_pct'])}")
    return m

print("=" * 80)
print(f"MODELO FINAL · {RUN_NAME} · teste: {len(Y_te)} imagens nunca vistas ({np.bincount(Y_te, minlength=2).tolist()})")
print("=" * 80)
print(f"treino (última época, com aumento): acc {hist_final.acc_treino.iloc[-1] * 100:.2f}% | loss {hist_final.loss_treino.iloc[-1]:.4f}")
print(f"treino (sem aumento, float)       : acc {((p_tr_float >= 0.5) == Y_all).mean() * 100:.2f}% | loss {log_loss(Y_all, p_tr_float):.4f}")
M_float = metrics("TESTE · float (Keras)", Y_te, p_te_float)
M_int8 = metrics("TESTE · int8 (ESP32)", Y_te, p_te_int8)
dif = (M_float["acuracia"] - M_int8["acuracia"]) * 100
print(f"\nperda com a quantização int8: {dif:+.2f} pontos de acurácia, AUC {M_float['auc'] - M_int8['auc']:+.3f}")

fig, ax = plt.subplots(1, 4, figsize=(21, 4.4))
h = hist_final
ax[0].plot(h.epoca, h.loss_treino, c="#c53030", label="loss treino")
ax[0].axhline(M_int8["loss"], c="#c53030", ls="--", label=f"loss teste int8 ({M_int8['loss']:.3f})")
a0 = ax[0].twinx(); a0.plot(h.epoca, h.acc_treino * 100, c="#2b6cb0", label="acc treino")
a0.axhline(M_int8["acuracia"] * 100, c="#2b6cb0", ls="--", label=f"acc teste int8 ({M_int8['acuracia'] * 100:.1f}%)")
a0.set_ylim(0, 100); ax[0].set_xlabel("época"); ax[0].set_title("Treino final × teste")
h1, l1 = ax[0].get_legend_handles_labels(); h2, l2 = a0.get_legend_handles_labels(); ax[0].legend(h1 + h2, l1 + l2, fontsize=7)
cm = np.array(M_int8["matriz_confusao"]); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
ax[1].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.0%}", ha="center", va="center", color="white" if cmn[i, j] > 0.5 else "black")
ax[1].set_xticks(range(2), LABELS); ax[1].set_yticks(range(2), LABELS)
ax[1].set_xlabel("previsto"); ax[1].set_ylabel("real"); ax[1].set_title("Matriz de confusão (int8, corte 0,5)")
for p, lab in [(p_te_float, "float"), (p_te_int8, "int8")]:
    ts = np.linspace(0, 1, 101); pos_ = Y_te == 1
    ax[2].plot([(p[~pos_] >= t).mean() for t in ts], [(p[pos_] >= t).mean() for t in ts], label=f"{lab} · AUC {auc_binary(pos_, p):.3f}")
ax[2].plot([0, 1], [0, 1], ":", c="gray"); ax[2].set_xlabel("limpas acionadas (FPR)"); ax[2].set_ylabel("sujas acionadas (recall)")
ax[2].set_title("Curva ROC"); ax[2].legend(); ax[2].grid(alpha=0.3)
ax[3].hist(p_te_int8[Y_te == 0], bins=20, range=(0, 1), alpha=0.6, label=LABELS[0])
ax[3].hist(p_te_int8[Y_te == 1], bins=20, range=(0, 1), alpha=0.6, label=LABELS[1])
ax[3].axvspan(LIMIAR_NAO_LIMPAR, max(LIMIAR_LIMPAR, LIMIAR_NAO_LIMPAR + 0.005), color="gold", alpha=0.3, label="revisão humana")
ax[3].set_xlabel("P(limpar) int8"); ax[3].set_title("Distribuição da confiança"); ax[3].legend()
plt.tight_layout(); plt.savefig(OUT / "metricas_teste.png", dpi=120); plt.show()

METRICAS = {"modelo": RUN_NAME, "treino": {"n": int(len(Y_all)), "acc_ultima_epoca": float(hist_final.acc_treino.iloc[-1]),
                                            "loss_ultima_epoca": float(hist_final.loss_treino.iloc[-1])},
            "teste_float": M_float, "teste_int8": M_int8,
            "validacao_cruzada": None if df_cv is None else df_cv.round(4).to_dict("records")}
(OUT / "metricas_modelo_final.json").write_text(json.dumps(METRICAS, indent=2, ensure_ascii=False, default=float), encoding="utf-8")

card = {
    "version": f"{ARCH}{'-' + SOIL_PRESET if ARCH == 'soilnet' else ''}-{IMG_SIZE}-{dt.date.today():%Y%m%d}", "task": "binario",
    "arch": ARCH, "soilnet": {"preset": SOIL_PRESET, "stem": SOIL_STEM, "blocks": SOIL_BLOCKS, "width": SOIL_WIDTH} if ARCH == "soilnet" else None,
    "class_folders": {"nao_aciona": neg, "aciona": pos}, "labels": list(LABELS), "positive_label": LABELS[1],
    "input_size": IMG_SIZE, "preprocessamento": "dataset tratado 224 (preprocess.py do notebook de tratamento) -> resize -> JPEG -> RGB565",
    "teste": TEST_PATH if TEST_PATH else f"split interno ({TEST_FRAC:.0%} das fotos reais, por grupo)", "n_teste": int(len(Y_te)),
    "jpeg_quality": JPEG_QUALITY, "rgb565": SIMULATE_RGB565,
    "limiar_nao_limpar": LIMIAR_NAO_LIMPAR, "limiar_limpar": LIMIAR_LIMPAR, "low_confidence_threshold": LIMIAR_LIMPAR,
    "parameters": int(final_model.count_params()), "tflite_bytes": len(blob), "tflite": info,
    "treino": {"origem": "do zero com 100% do dataset de treino", "n": int(len(Y_all)), "sinteticas": int(SYN.sum()),
               "epocas": EPOCHS_SOILNET if ARCH == "soilnet" else [EPOCHS_HEAD, EPOCHS_FINETUNE],
               "aumento": {"rot90_copias": AUG_ROT90_COPIES, "graus_aleatorios": AUG_RANDOM_DEG, "espelhamento": AUG_FLIP}},
    "avaliacao_teste_int8": {k: M_int8[k] for k in ("acuracia", "loss", "f1_macro", "auc", "recall_limpar",
                                                     "precisao_limpar", "especificidade_nao_limpar", "decisao", "ic95")},
    "avaliacao_teste_float": {k: M_float[k] for k in ("acuracia", "loss", "f1_macro", "auc")},
}
(OUT / "model_card.json").write_text(json.dumps(card, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print("salvos:", OUT / "metricas_modelo_final.json", "|", OUT / "model_card.json")

## 12. Onde o modelo int8 erra no teste

In [ ]:
pred8 = (p_te_int8 >= 0.5).astype(int)
err = np.where(pred8 != Y_te)[0]
err = err[np.argsort(-np.abs(p_te_int8[err] - 0.5))][:16]
print(f"{(pred8 != Y_te).sum()} erros em {len(Y_te)} imagens (corte 0,5)")
if len(err):
    cols = 8; rows = math.ceil(len(err) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2.4 * rows), squeeze=False)
    for ax, i in zip(axes.flat, err):
        ax.imshow(X_te[i]); ax.set_title(f"real {LABELS[Y_te[i]][:9]}\nP(limpar) {p_te_int8[i]:.2f}", fontsize=7)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout(); plt.show()
pd.DataFrame({"arquivo": [R_te[i]["path"].name for i in np.where(pred8 != Y_te)[0]],
              "real": [LABELS[Y_te[i]] for i in np.where(pred8 != Y_te)[0]],
              "p_limpar_int8": p_te_int8[pred8 != Y_te].round(3)}).to_csv(OUT / "erros_teste.csv", index=False)

## 13. Pacote para download

In [ ]:
pacote = shutil.make_archive(str(OUT), "zip", OUT)
print("pacote com todos os arquivos:", pacote)
print("conteúdo:", sorted(p.name for p in OUT.iterdir() if p.is_file()))
print("\nATENÇÃO: confira no firmware INPUT_SIZE =", IMG_SIZE, "| 2 rótulos", list(LABELS),
      f"| regra da bomba P(limpar) >= {LIMIAR_LIMPAR:.2f}")
if IN_COLAB:
    from google.colab import files
    files.download(pacote)

## 14. (Opcional) Exportar o modelo float em ONNX

Só com `EXPORT_ONNX = True`. Instala o `tf2onnx`, que pode alterar versões de pacotes do ambiente; por isso fica
**por último**, depois que todos os arquivos do ESP32 já foram salvos. O ESP32 continua usando o `.tflite`.

In [ ]:
if EXPORT_ONNX:
    import subprocess, importlib, importlib.util
    try:
        if importlib.util.find_spec("tf2onnx") is None or importlib.util.find_spec("onnxruntime") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tf2onnx", "onnxruntime"], check=True)
            importlib.invalidate_caches()
        onnx_path = OUT / "model.onnx"
        r = subprocess.run([sys.executable, "-m", "tf2onnx.convert", "--saved-model", str(OUT / "saved_model"),
                            "--output", str(onnx_path), "--opset", "17"], capture_output=True, text=True)
        if r.returncode != 0 or not onnx_path.exists():
            raise RuntimeError(r.stderr[-800:])
        import onnxruntime as ort
        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        p_onnx = sess.run(None, {sess.get_inputs()[0].name: X_te[:32].astype("float32")})[0][:, 1]
        print(f"ONNX: {onnx_path} ({onnx_path.stat().st_size / 1024:.0f} KB) | diferença máx. x Keras: "
              f"{np.abs(p_onnx - p_te_float[:32]).max():.2e}")
        print("entrada: float32 NHWC [N,", IMG_SIZE, IMG_SIZE, "3], RGB 0..255 | saída: [P(não limpar), P(limpar)]")
    except Exception as e:
        print("exportação ONNX falhou (os arquivos do ESP32 não foram afetados):", e)
else:
    print("EXPORT_ONNX = False")

## 15. (Opcional) Classificar uma foto avulsa com o modelo int8

A foto passa pelo mesmo caminho do treino. Se ela **não** estiver tratada, aplique antes o `preprocess.py` do
notebook de tratamento (enquadramento + 224).

In [ ]:
FOTO = r"C:\caminho\para\uma_foto.jpg"
if IN_COLAB and not Path(FOTO).exists():
    from google.colab import files
    FOTO = "/content/" + next(iter(files.upload()))
if Path(FOTO).exists():
    x = device_like(FOTO)[None]
    ps = float(tflite_predict(blob, x)[0])
    print(f"P({LABELS[1]}) = {ps:.3f} -> {decide(np.array([ps]), LIMIAR_NAO_LIMPAR, LIMIAR_LIMPAR)[0]}")
    plt.imshow(x[0]); plt.axis("off"); plt.show()
else:
    print("ajuste FOTO para o caminho de uma imagem")

## 16. Observações

1. **Comece rápido:** `QUICK=True` confere que tudo roda; depois `N_FOLDS_TO_RUN=1` ou `2`.
2. **Treino × teste por fold:** treino muito acima do teste = overfitting (menos épocas, mais `DROPOUT`, `SOIL_PRESET="1x"`);
   treino abaixo de ~80% = subajuste (mais épocas, `"2x"` ou `mbv1_025`).
3. **int8 × float:** se a acurácia int8 cair mais de 2–3 pontos em relação à float, teste `mbv1_025` ou mais imagens no
   conjunto representativo da quantização.
4. **Câmera no campo:** o modelo aprendeu com fotos **enquadradas na placa**. Posicione a câmera do ESP32 para a placa
   ocupar a maior parte da imagem, ou aplique o mesmo enquadramento no backend antes de enviar.